# RwandaCropGuard – Kaggle GPU experiments

40 full experiments (10 x DQN, REINFORCE, A2C, PPO) using TorchRL.

Settings: GPU + Internet ON. 40000 steps per experiment (~650-700 episodes).

Repo: https://github.com/francis-collab/francis_mutabazi_rl_summative.git


## 1. Clone repository and install dependencies


In [ ]:
import os
from pathlib import Path

os.chdir("/kaggle/working")

# Fresh clone
!rm -rf francis_mutabazi_rl_summative
!git clone https://github.com/francis-collab/francis_mutabazi_rl_summative.git

os.chdir("/kaggle/working/francis_mutabazi_rl_summative")
print("CWD:", os.getcwd())
print("Files:", sorted(os.listdir("."))[:20])

!pip -q install "torchrl>=0.7.0" "tensordict>=0.7.0" "gymnasium>=0.29.0" matplotlib pandas tqdm seaborn

import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


## 2. Hyperparameter grids (10 per algorithm)

Defined in source and used by run_experiments.py:

**DQN** (training/dqn_training.py):

| Exp | LR | Gamma | Batch | eps start | eps end | eps decay |
|-----|-----|-------|-------|-----------|---------|-----------|
| exp01 | 1e-3 | 0.99 | 64 | 1.0 | 0.05 | 12000 |
| exp02 | 3e-4 | 0.99 | 64 | 1.0 | 0.05 | 12000 |
| exp03 | 1e-4 | 0.99 | 64 | 1.0 | 0.05 | 12000 |
| exp04 | 1e-3 | 0.95 | 64 | 1.0 | 0.05 | 12000 |
| exp05 | 1e-3 | 0.99 | 128 | 1.0 | 0.05 | 12000 |
| exp06 | 1e-3 | 0.99 | 32 | 1.0 | 0.05 | 12000 |
| exp07 | 1e-3 | 0.99 | 64 | 1.0 | 0.01 | 12000 |
| exp08 | 1e-3 | 0.99 | 64 | 1.0 | 0.05 | 6000 |
| exp09 | 1e-3 | 0.99 | 64 | 1.0 | 0.05 | 20000 |
| exp10 | 3e-4 | 0.99 | 128 | 1.0 | 0.05 | 12000 |

**REINFORCE / A2C / PPO** (training/pg_training.py):

| Exp | LR | Gamma | Entropy | Rollout |
|-----|-----|-------|---------|---------|
| *_exp01 | 3e-4 | 0.99 | 0.01 | 256 |
| *_exp02 | 1e-4 | 0.99 | 0.01 | 256 |
| *_exp03 | 1e-3 | 0.99 | 0.01 | 256 |
| *_exp04 | 3e-4 | 0.95 | 0.01 | 256 |
| *_exp05 | 3e-4 | 0.99 | 0.001 | 256 |
| *_exp06 | 3e-4 | 0.99 | 0.05 | 256 |
| *_exp07 | 3e-4 | 0.99 | 0.01 | 128 |
| *_exp08 | 3e-4 | 0.99 | 0.01 | 512 |
| *_exp09 | 5e-4 | 0.99 | 0.01 | 256 |
| *_exp10 | 3e-4 | 0.995 | 0.01 | 256 |

Each run uses 40000 environment steps.


In [ ]:
from training.dqn_training import dqn_hyperparameter_grid
from training.pg_training import pg_hyperparameter_grid

print("=== DQN (10) ===")
for c in dqn_hyperparameter_grid("cpu", 40_000):
    print(c.exp_name, "lr", c.lr, "gamma", c.gamma, "batch", c.batch_size,
          "eps", c.eps_start, "->", c.eps_end, "decay", c.eps_decay_steps)

for algo in ("reinforce", "a2c", "ppo"):
    print("\n===", algo.upper(), "(10) ===")
    for c in pg_hyperparameter_grid(algo, "cpu", 40_000):
        print(c.exp_name, "lr", c.lr, "gamma", c.gamma,
              "entropy", c.entropy_coef, "rollout", c.rollout_steps)


## 3. Environment demo


In [ ]:
!python main.py --mode demo


## 4. Run all 40 experiments


In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
!python run_experiments.py --device {device} --total-steps 40000


## 5. Results summary tables


In [ ]:
import json
from pathlib import Path
import pandas as pd
summary_path = Path("logs/full_experiment_summary.json")
if summary_path.is_file():
    summary = json.loads(summary_path.read_text())
    for algo, rows in summary.items():
        print("\n===", algo.upper(), "===")
        print(pd.DataFrame(rows).to_string(index=False))
else:
    print("Summary not found")


## 6. Display training plots


In [ ]:
from IPython.display import Image, display
from pathlib import Path
plots = sorted(Path("models").rglob("training_plot.png"))
print("Found", len(plots), "plots")
for p in plots:
    print(p)
    display(Image(str(p)))


## 7. Zip artefacts for download


In [ ]:
!zip -r /kaggle/working/rwanda_cropguard_results.zip models logs assets
!ls -lh /kaggle/working/rwanda_cropguard_results.zip
print("Download from Output panel")
